# EXAONE 4.0 Final Quantization Factory

**목표**: 단 한 번의 실행으로 여러가지 최적화된 양자화 모델(FP8, INT8, INT4 변형들)을 생성하고 제출용 Zip 파일을 생성합니다.

**특징**:
- **Native Environment**: Colab 기본 환경 유지 + `llm-compressor` Add-on
- **Architecture Safe**: EXAONE 4.0의 `QK-Reorder-LN` 구조 보호 (Sensitive Layer Ignore)
- **Auto Submission**: 각 전략 완료 즉시 결과물 압축

---

In [ ]:
pip install llmcompressor

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 282.1/282.1 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 132.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.3/566.3 kB 53.6 MB/s eta 0:00:00
  Attempting uninstall: nvidia-ml-py
    Found existing installation: nvidia-ml-py 13.590.48
    Uninstalling nvidia-ml-py-13.590.48:
      Successfully uninstalled nvidia-ml-py-13.590.48
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.3.4
    Uninstalling huggingface_hub-1.3.4:
      Successfully uninstalled huggingface_hub-1.3.4
  Attempting uninstall: transformers
    Found existing installat

In [ ]:
# Google Drive & Hugging Face Setup
import os
from pathlib import Path
from google.colab import drive
from google.colab import userdata
from huggingface_hub import login

# 1. Drive Mount
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 2. Path Config
from pathlib import Path
DRIVE_ROOT = Path("/content/drive/MyDrive")
WORK_DIR = DRIVE_ROOT / "EXAONE_Colab"
BASE_PATH = WORK_DIR

# Model Source
MODEL_ID_PATH = DRIVE_ROOT / "base_model"
if not MODEL_ID_PATH.exists():
    print(f"⚠️ Local model not found at {MODEL_ID_PATH}. Using HF Hub.")
    MODEL_ID = "LGAI-EXAONE/EXAONE-4.0-1.2B"
else:
    MODEL_ID = str(MODEL_ID_PATH)
    print(f"✅ Using Local Model: {MODEL_ID}")

# 3. HF Login
try:
    HF_TOKEN = "hf_"
    login(token=HF_TOKEN)
except:
    print("⚠️ HF_TOKEN isn't validate.")

✅ Using Local Model: /content/drive/MyDrive/base_model


In [ ]:
import os
import shutil
from datasets import load_dataset, load_from_disk, concatenate_datasets

# 💾 저장 경로
CALIB_SAVE_DIR = "calib_dataset_processed"


# =========================================================
# (All Splits + MANTA Sampling)
# =========================================================
if not os.path.exists(CALIB_SAVE_DIR) or 'calib_ds' not in locals():
    print("\n⚡ 데이터 다운로드 및 전처리를 시작합니다...")

    # [Helper] Split 구분 없이 모든 데이터를 합쳐서 로드하는 함수
    def load_all_splits(dataset_name):
        print(f"   📥 Loading {dataset_name} (All splits)...")
        ds_dict = load_dataset(dataset_name, trust_remote_code=False)
        # DatasetDict(여러 split)인 경우 하나로 합침
        if hasattr(ds_dict, 'values'):
            return concatenate_datasets(list(ds_dict.values()))
        return ds_dict

    # 1) Ko-LongRAG (전체)
    ds_longrag = load_all_splits("LGAI-EXAONE/Ko-LongRAG")

    # 2) MANTA-1M (전체 로드 후 -> 샘플링)
    # MANTA는 워낙 커서 로드만 하고 바로 샘플링하는 것이 효율적
    ds_manta_full = load_dataset("LGAI-EXAONE/MANTA-1M", split="train", trust_remote_code=False)

    # 3) KMMLU-Pro (전체)
    ds_kmmlu_pro = load_all_splits("LGAI-EXAONE/KMMLU-Pro")

    # 4) KMMLU-Redux (전체)
    ds_kmmlu_redux = load_all_splits("LGAI-EXAONE/KMMLU-Redux")

    LGAI-EXAONE/KoMT-Bench

    print(f"   - Ko-LongRAG Total: {len(ds_longrag)}")
    print(f"   - MANTA Original : {len(ds_manta_full)}")
    print(f"   - KMMLU-Pro Total : {len(ds_kmmlu_pro)}")
    print(f"   - KMMLU-Redux Total: {len(ds_kmmlu_redux)}")

    # ---- [MANTA 샘플링] ----
    # 텍스트 변환(Map) 전에 샘플링해야 속도가 빠릅니다.
    MANTA_SAMPLE_SIZE = 4096  # 필요에 따라 조절 (예: 2048 ~ 10000)
    print(f"   ✂️ MANTA 데이터가 너무 커서 {MANTA_SAMPLE_SIZE}개만 랜덤 샘플링합니다.")
    ds_manta = ds_manta_full.shuffle(seed=42).select(range(min(len(ds_manta_full), MANTA_SAMPLE_SIZE)))

    # ---- 포맷 변환 함수들 ----
    def format_options(options):
        if isinstance(options, list):
            return "\n".join(f"{chr(65+i)}. {opt}" for i, opt in enumerate(options))
        return str(options)

    def to_text_longrag(example):
        return {"text": example["context"] + "\n\nQ: " + example["question"] + "\nA: " + example["answer"]}

    def to_text_manta(example):
        parts = [f"{turn['role']}: {turn['content']}" for turn in example["conversations"]]
        return {"text": "\n".join(parts)}

    def to_text_kmmlu(example):
        q = example["question"]
        opts = format_options(example["options"])
        sol = example["solution"]
        return {"text": q + "\n\n" + opts + "\n\n정답: " + str(sol)}

    # ---- 실제 변환 수행 ----
    print("   🔨 텍스트 포맷 변환 중...")
    longrag_text = ds_longrag.map(to_text_longrag, remove_columns=ds_longrag.column_names)
    manta_text   = ds_manta.map(to_text_manta,   remove_columns=ds_manta.column_names)
    kmpro_text   = ds_kmmlu_pro.map(to_text_kmmlu, remove_columns=ds_kmmlu_pro.column_names)
    kmredux_text = ds_kmmlu_redux.map(to_text_kmmlu, remove_columns=ds_kmmlu_redux.column_names)

    # 최종 병합
    calib_ds = concatenate_datasets([longrag_text, manta_text, kmpro_text, kmredux_text]).shuffle(seed=42)

    # ---- [저장] ----
    print(f"   💾 디스크에 저장 중... ({CALIB_SAVE_DIR})")
    calib_ds.save_to_disk(CALIB_SAVE_DIR)

    # (선택) JSONL 백업
    calib_ds.to_json(os.path.join(CALIB_SAVE_DIR, "calib_preview.jsonl"), force_ascii=False)
    print("   ✅ 생성 및 저장 완료.")

# =========================================================
# 최종 확인
# =========================================================
print(f"\n📊 최종 Calibration 데이터셋 크기: {len(calib_ds)}")
for i in range(2):
    print(f"   Sample {i} length: {len(calib_ds[i]['text'])} chars")


⚡ 데이터 다운로드 및 전처리를 시작합니다...
   📥 Loading LGAI-EXAONE/Ko-LongRAG (All splits)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/19.2M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/600 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/2587 [00:00<?, ? examples/s]

   💾 디스크에 저장 중... (calib_dataset_processed)


Saving the dataset (0/1 shards):   0%|          | 0/10105 [00:00<?, ? examples/s]

Creating json from Arrow format:   0%|          | 0/11 [00:00<?, ?ba/s]

   ✅ 생성 및 저장 완료.

📊 최종 Calibration 데이터셋 크기: 10105
   Sample 0 length: 163 chars
   Sample 1 length: 5049 chars


In [ ]:
from random import sample

N_SHOW = 20  # 보고 싶은 샘플 개수

print(f"\n📊 최종 Calibration 데이터셋 크기: {len(calib_ds)}")

# 1) 랜덤 인덱스 뽑기
idxs = sample(range(len(calib_ds)), k=min(N_SHOW, len(calib_ds)))

for i, idx in enumerate(idxs):
    text = calib_ds[idx]["text"]
    preview = text[:400] + "..." if len(text) > 400 else text
    print(f"\n🧪 Random Sample {i} (idx={idx})")
    print(f"   길이: {len(text)} chars")
    print(f"   내용:\n{preview}")
    print("-" * 80)



📊 최종 Calibration 데이터셋 크기: 10105

🧪 Random Sample 0 (idx=5638)
   길이: 120 chars
   내용:
인탈리오(intalio), 셰비(chevee), 큐벳(cuvette) 등은 다음 중 어느 기법인가?

A. 귀금속판 돋아올리기
B. 귀금속 구슬모아 용접하기
C. 보석구슬 구멍뚫기
D. 보석표면 조각하기

정답: 4
--------------------------------------------------------------------------------

🧪 Random Sample 1 (idx=1453)
   길이: 4591 chars
   내용:
user: Discuss the multifaceted role of phosphorus in modern agriculture and industry, highlighting its extraction methods and environmental implications. In your essay, analyze how the demand for phosphorus-rich fertilizers impacts both crop yields and ecological systems, and propose potential sustainable practices to mitigate adverse effects while ensuring food security.
assistant: ### The Multif...
--------------------------------------------------------------------------------

🧪 Random Sample 2 (idx=5340)
   길이: 17830 chars
   내용:
Title: 1975년 AFC 청소년 축구 선수권 대회 Text:1975년 AFC 청소년 축구 선수권 대회는 1975년 4월 4일부터 20일까지 쿠웨이트 쿠웨이트시티에서 개최된 17번째 AFC 청소년 축구 

In [ ]:
# [Step 4]

COMMON_CONFIG = {
    "targets": ["Linear"],
    "ignore": ["lm_head", "norm", "rotary_emb"],
    "num_calib": min(len(calib_ds), 512),
    "max_seq": 2048,
}

STRATEGIES = [
    #  AWQ W4A16
    {
        "name": "10_AWQ_W4A16",
        "output_dir": "exaone-awq-w4a16",
        "modifier_type": "AWQ",  # AWQModifier 사용 지시
        "recipe_args": {
            "scheme": "W4A16",
            "targets": COMMON_CONFIG["targets"],
            "ignore": COMMON_CONFIG["ignore"],
        },
        # "use_smoothquant": False,
        # "use_spinquant": True,
    }
]

print(f"📋 Configured {len(STRATEGIES)} Mixed Strategies (AWQ Focus for L4).")

📋 Configured 3 Mixed Strategies (AWQ Focus for L4).


In [ ]:
import gc
import torch
import shutil
import os
from pathlib import Path
import time

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# === LLM Compressor 최신 API ===
from llmcompressor.modifiers.smoothquant import SmoothQuantModifier
from llmcompressor.modifiers.quantization import GPTQModifier
# AWQ Modifier 임포트 필수
from llmcompressor.modifiers.awq import AWQModifier
from llmcompressor import oneshot

total_start = time.time()
success_count = 0

# ==========================================
# Step 5: Mixed 전략용 Batch Execution
# ==========================================

for idx, strategy in enumerate(STRATEGIES):
    print(f"\n{'='*70}")
    print(f"🚀 [{idx+1}/{len(STRATEGIES)}] {strategy['name']}")
    print('='*70)

    # 1. 경로 설정
    STRATEGY_ROOT = BASE_PATH / strategy['output_dir']
    MODEL_SAVE_DIR = STRATEGY_ROOT / "model"

    SAVE_DIR_STR = str(MODEL_SAVE_DIR)
    ROOT_DIR_STR = str(STRATEGY_ROOT)

    # 이미 성공한 전략은 건너뛰기
    if os.path.exists(ROOT_DIR_STR) and os.path.exists(f"{ROOT_DIR_STR}.SUCCESS"):
        print(f"   ✅ Skip: {strategy['name']}")
        success_count += 1
        continue

    # 기존 폴더 정리
    if os.path.exists(ROOT_DIR_STR):
        shutil.rmtree(ROOT_DIR_STR)
    os.makedirs(SAVE_DIR_STR)

    # 🔧 Recipe 구성
    recipe = []

    # # 1) SmoothQuant (Pre-processing)
    # if strategy.get("use_smoothquant", False):
    #     sq_strength = strategy.get("smooth_strength", 0.8)
    #     recipe.append(SmoothQuantModifier(smoothing_strength=sq_strength))
    #     print(f"   ➕ SmoothQuantModifier(strength={sq_strength})")

    # # # 2) SpinQuant (Placeholder)
    # # if strategy.get("use_spinquant", False):
    # #     print(f"   ⚠️ SpinQuant 활성화됨 (Note: 현재 로직에서는 별도 Modifier가 아닌 전처리 로직 필요 가능성 있음)")

    # 3) Main Quantizer (AWQ vs GPTQ 분기 처리)
    mod_type = strategy.get("modifier_type", "GPTQ")
    base_args = strategy["recipe_args"].copy()

    # 공통 타겟 설정
    base_args["targets"] = base_args.get("targets", COMMON_CONFIG["targets"])
    base_args["ignore"] = base_args.get("ignore", COMMON_CONFIG["ignore"])

    if mod_type == "AWQ":
        # === AWQ Modifier ===
        print(f"   📝 Applying AWQModifier: scheme={base_args.get('scheme')} | group_size={base_args.get('group_size')}")
        recipe.append(AWQModifier(**base_args))

    else:
        # === GPTQ Modifier ===
        # GPTQModifier는 'group_size' 인자를 받지 않으므로 제거
        if "group_size" in base_args:
            print(f"   ⚠️ Removing 'group_size' from GPTQ args (Not supported in GPTQModifier)")
            base_args.pop("group_size")

        base_args["block_size"] = base_args.get("block_size", 128)

        print(f"   📝 Applying GPTQModifier: scheme={base_args.get('scheme')} | block_size={base_args['block_size']}")
        recipe.append(GPTQModifier(**base_args))


    start_time = time.time()
    try:
        gc.collect(); torch.cuda.empty_cache()
        print("   ⚡ oneshot 시작...")

        oneshot(
            model=MODEL_ID,
            dataset=calib_ds,
            recipe=recipe,
            max_seq_length=COMMON_CONFIG["max_seq"],
            num_calibration_samples=COMMON_CONFIG["num_calib"],
            output_dir=SAVE_DIR_STR,
        )

        elapsed = time.time() - start_time
        print(f"   ✅ 완료! {elapsed/60:.1f}분")

        # 성공 마커 및 압축
        Path(f"{ROOT_DIR_STR}.SUCCESS").touch()

        zip_filename = f"submit_{strategy['name']}"
        zip_path = str(BASE_PATH / zip_filename)

        shutil.make_archive(
            base_name=zip_path,
            format="zip",
            root_dir=ROOT_DIR_STR,
            base_dir="model"
        )

        print(f"   📦 Saved: {zip_path}.zip")
        success_count += 1

    except Exception as e:
        print(f"   ❌ Error: {e}")
        # 에러 스택트레이스 상세 출력을 원하면 아래 주석 해제
        # import traceback; traceback.print_


🚀 [1/3] 1_AWQ_W4A16_G128
   ✅ Skip: 1_AWQ_W4A16_G128

🚀 [2/3] 2_W8A8_INT8_SQ
   ➕ SmoothQuantModifier(strength=0.8)
   📝 Applying GPTQModifier: scheme=W8A8 | block_size=128
   ⚡ oneshot 시작...


Tokenizing:   0%|          | 0/10105 [00:00<?, ? examples/s]

2026-02-04T05:54:20.770266+0000 | reset | INFO - Compression lifecycle reset
2026-02-04T05:54:20.775889+0000 | from_modifiers | INFO - Creating recipe from modifiers
2026-02-04T05:54:20.776606+0000 | _infer_mappings_from_model | INFO - No SmoothQuantModifier.mappings provided, inferring from model...
2026-02-04T05:54:20.777115+0000 | get_layer_mappings_from_architecture | INFO - Architecture Exaone4ForCausalLM not found in mappings. Using default mappings: [LayerMap(balance_layers=['re:.*q_proj', 're:.*k_proj', 're:.*v_proj'], smooth_layers='re:.*input_layernorm'), LayerMap(balance_layers=['re:.*gate_proj', 're:.*up_proj'], smooth_layers='re:.*post_attention_layernorm')]
   ❌ Error: Error resolving mappings for given architecture.Please refer to the README at https://github.com/vllm-project/llm-compressor/tree/main/src/llmcompressor/modifiers/smoothquant for more information.

🚀 [3/3] 3_SpinQuant_W4A16
   ⚠️ SpinQuant 활성화됨 (Note: 현재 로직에서는 별도 Modifier가 아닌 전처리 로직 필요 가능성 있음)
   📝 Applying

Tokenizing:   0%|          | 0/10105 [00:00<?, ? examples/s]

2026-02-04T05:55:24.035190+0000 | reset | INFO - Compression lifecycle reset
2026-02-04T05:55:24.041193+0000 | from_modifiers | INFO - Creating recipe from modifiers
2026-02-04T05:55:24.080815+0000 | initialize | INFO - Compression lifecycle initialized for 1 modifiers
2026-02-04T05:55:24.081481+0000 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `GPTQModifier`


...

2026-02-04T06:04:21.732232+0000 | compress | METRIC - time 1.22s
2026-02-04T06:04:21.734593+0000 | compress | METRIC - error 5496.91
2026-02-04T06:04:21.735587+0000 | compress | METRIC - GPU 0 | usage: 2.63% | total memory: 85 GB
2026-02-04T06:04:21.736168+0000 | compress | METRIC - Compressed module size: 8.486912 MB
2026-02-04T06:04:21.737183+0000 | compress_modules | INFO - Quantizing model.layers.29.self_attn.k_proj using 512 samples
2026-02-04T06:04:22.937009+0000 | compress | METRIC - time 1.20s
2026-02-04T06:04:22.939322+0000 | compress | METRIC - error 1559.24
2026-02-04T06:04:22.940079+0000 | compress | METRIC - GPU 0 | usage: 2.63% | total memory: 85 GB
2026-02-04T06:04:22.940656+0000 | compress | METRIC - Compressed module size: 2.121728 MB
2026-02-04T06:04:22.941521+0000 | compress_modules | INFO - Quantizing model.layers.29.self_attn.v_proj using 512 samples
2026-02-04T06:04:24.117313+0000 | compress | METRIC - time 1.18s
2026-02-04T06:04:24.119600+0000 | compress | METRIC

(31/31): Propagating: 100%|██████████| 512/512 [00:01<00:00, 419.22it/s]

2026-02-04T06:04:36.581131+0000 | finalize | INFO - Compression lifecycle finalized for 1 modifiers
2026-02-04T06:04:36.643514+0000 | get_model_compressor | INFO - skip_sparsity_compression_stats set to True. Skipping sparsity compression statistic calculations. No sparsity compressor will be applied.



Compressing model: 210it [00:02, 97.03it/s]


   ✅ 완료! 10.4분
   📦 Saved: /content/drive/MyDrive/EXAONE_Colab/submit_3_SpinQuant_W4A16.zip
